In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split
from CNN import run_cv
from plotting import plot_heatmap

# Load data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Select 10,000 images with stratified split
Xtr, _, ytr, _ = train_test_split(
    x_train, y_train,
    train_size=10000,
    stratify=y_train,
    random_state=2026
)

In [ ]:
learning_rates = [1e-2, 1e-3, 1e-4]
dropouts = [0.0, 0.3, 0.6]

In [ ]:
losses = np.zeros((len(learning_rates), len(dropouts)))

for i, lr in enumerate(learning_rates):
    for j, dropout in enumerate(dropouts):

        loss, best_epoch = run_cv(Xtr, ytr, lr=lr, dropout=dropout)
        losses[i, j] = loss

        print(f"LR={lr}, Dropout={dropout}: "
              f"val loss={loss:.4f}, best epoch={best_epoch}")

In [ ]:
plot_heatmap(
    losses, dropouts, learning_rates,
    "Dropout", "Learning rate",
    title="CNN min mean val loss, 5-fold CV, 10.000 tr.im., max 30 ep."
)
plt.savefig("../figures/CNN_CV1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
lr = 1e-4
dropout = 0.6
conv_layers = [1, 2]
kernel_size = [3, 5]

In [ ]:
losses = np.zeros((len(conv_layers), len(kernel_size)))

for i, cl in enumerate(conv_layers):
    for j, ks in enumerate(kernel_size):

        loss, best_epoch = run_cv(Xtr, ytr, lr=lr, dropout=dropout, kernel_size=ks, conv_layers=cl, epochs=50)
        losses[i, j] = loss

        print(f"Conv. layers={cl}, Kernel size={ks}: "
              f"val loss={loss:.4f}, best epoch={best_epoch}")

In [ ]:
plot_heatmap(
    losses, kernel_size, conv_layers,
    "Kernel size", "Conv. layers",
    title="CNN min mean val loss, 5-fold CV, 10.000 tr.im., max 50 ep."
)

plt.savefig("../figures/CNN_CV_architecture.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
lr = 1e-4
dropout = 0.6
conv_layers = 2
kernel_size = 3
dense_units = [128, 256, 512, 1024]

In [ ]:
losses = []
best_epochs = []

for units in dense_units:
    loss, best_epoch = run_cv(Xtr, ytr, lr=lr, dropout=dropout, kernel_size=kernel_size, conv_layers=conv_layers, dense_units=units, epochs=50)

    losses.append(loss)
    best_epochs.append(best_epoch)

    print(f"Dense units={units}: val loss={loss:.4f}, best epoch={best_epoch}")

In [ ]:
from tensorflow import keras
from CNN import make_model
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# Best hyperparameters from CV
lr = 1e-4
dropout = 0.6
kernel_size = 3
conv_layers = 2
dense_units = 1024
epochs = 29

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Load all 60,000 training and 10,000 test images
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalize using training statistics only
mean, std = x_train.mean(), x_train.std()
X_train = ((x_train - mean) / std).astype("float32")[..., None]
X_test = ((x_test - mean) / std).astype("float32")[..., None]

# Make final model
keras.utils.set_random_seed(2026)
model = make_model(lr, dropout, kernel_size, conv_layers, dense_units)

In [ ]:
#Train final model
model.fit(X_train, y_train, epochs=epochs, batch_size=64, verbose=1)

In [ ]:
# Evaluate on test set ONCE

probs = keras.activations.softmax(model.predict(X_test, verbose=0)).numpy()
y_pred = probs.argmax(axis=1)

# Accuracy ± standard error
acc = accuracy_score(y_test, y_pred)
se = np.sqrt(acc * (1 - acc) / len(y_test))

print(f"Test accuracy: {acc:.4f} ± {se:.4f}")
print(f"Parameter count: {model.count_params():,}")

# Per-class F1 scores
print("\nPer-class metrics:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# Confusion matrix and one-vs-rest ROC/AUC
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names,
    ax=ax[0], xticks_rotation=90, cmap="Blues"
)
ax[0].set_title("Confusion matrix")

for i, name in enumerate(class_names):
    RocCurveDisplay.from_predictions(
        (y_test == i).astype(int), probs[:, i],
        name=name, ax=ax[1]
    )

ax[1].set_title("One-vs-rest ROC curves")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("../figures/CNN_CV_final_results.png",
            dpi=300, bbox_inches="tight")
plt.show()